In [25]:
from datetime import datetime
import os
import sys

import pandas as pd
import numpy as np
from pymongo import MongoClient

sys.path.append(os.path.abspath('..'))

from app.models.movie import Movie


def load_env_file(env_path: str) -> None:
    if not os.path.exists(env_path):
        return

    with open(env_path, encoding='utf-8') as env_file:
        for raw_line in env_file:
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue

            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip())


load_env_file('../.env')

mongo_url = os.environ.get('MONGODB_URL', 'mongodb://localhost:27017')
database_name = os.environ.get('DATABASE_NAME', 'recommendation_engine')
client = MongoClient(mongo_url)
db = client[database_name]


movies_df = pd.read_csv('../../data/processed/movies.csv')
sentiment_df = pd.read_csv('../../data/processed/item_sentiment_scores.csv')

# Merge sentiment labels
movies_df = movies_df.merge(
    sentiment_df[['item_id', 'label']],
    on='item_id',
    how='left'
 )

# Rename columns
movies_df = movies_df.rename(
    columns={
        'label': 'sentiment_label',
        'description': 'overview', 
        'categories': 'genres', 
        'average_rating': 'avg_rating'
    }
 )

# Fill missing values before creating Pydantic models
movies_df['title'] = movies_df['title'].fillna('Untitled Movie')
movies_df['overview'] = movies_df['overview'].fillna('No overview available')
movies_df['avg_rating'] = movies_df['avg_rating'].fillna(0.0)
movies_df['rating_number'] = movies_df['rating_number'].fillna(0)
movies_df['sentiment_label'] = movies_df['sentiment_label'].fillna('Niche Appeal')

# Convert genres
if 'genres' in movies_df.columns:
    movies_df['genres'] = movies_df['genres'].apply(
        lambda x: x.split('|') if isinstance(x, str) else []
    )
else:
    movies_df['genres'] = [[] for _ in range(len(movies_df))]

# Replace NaN
movies_df = movies_df.replace({np.nan: None})

# Create Pydantic models
movies = [
    Movie(
        **{
            **movie,
            'created_at': datetime.utcnow()
        }
    )
    for movie in movies_df.to_dict(orient='records')
]

# Batch insert all movies into MongoDB
movie_docs = [
    movie.model_dump(exclude_none=True)
    for movie in movies
]
result = db.movies.insert_many(movie_docs)

print(f"Inserted {len(result.inserted_ids)} movies successfully!")

Inserted 99223 movies successfully!
